In [ ]:
import torch

if torch.cuda.is_available():
    print("✅ GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ GPU is not available. Using CPU instead.")

In [2]:
%%capture
!pip install transformers
!pip install librosa
!pip install jiwer
!pip install evaluate
!pip install wandb
!pip install numpy==2.1.0
!pip install scipy==1.11.4
!pip install librosa==0.10.1
!pip install numba==0.58.1
!pip install datasets>=2.14.0
!pip install accelerate>=0.26.0
!pip install typing_extensions --upgrade

In [3]:
# !pip install huggingface_hub --quiet
import os
os.environ["HF_HOME"]="/nvme/cache" # set cache dir for hf datasets

from huggingface_hub import login
login(token="INSERT_YOUR_HUGGING_FACE_TOKEN_HERE")

In [ ]:
import wandb

wandb.login(key="INSERT_YOUR_TOKEN_HERE")

In [5]:
%%capture
!apt install git-lfs

In [ ]:
import shutil

# Check space where the container is running (usually '/')
total, used, free = shutil.disk_usage("/nvme") # use /nvme and not /

print(f"Total Space: {total // (2**30)} GB")
print(f"Used Space:  {used // (2**30)} GB")
print(f"Free Space:  {free // (2**30)} GB")

In [ ]:
from datasets import load_dataset, concatenate_datasets, Audio

ds = load_dataset("kaarthu2003/Telugu_Audio_5s_Trimmed")
train_dataset = ds["train"]
val_dataset = ds["validation"]

In [ ]:
# Print confirmation
print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")

# Sample peek
print("\nSample example:")
print(train_dataset[0])

In [ ]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

show_random_elements(train_dataset.remove_columns(["audio"]))

In [10]:
telugu_special_unwanted_characters = [
    'ఁ',  # Chandrabindu
    'ౄ',  # Vocalic RR
    'ౢ',  # Vocalic L
    'ౣ',  # Vocalic LL
    'ౠ',  # Long Vocalic RR
    'ఽ',  # Avagraha
    '౦', '౧', '౨', '౩', '౪', '౫', '౬', '౭', '౮', '౯',  # Telugu digits
    'ఀ',  # Telugu Sign Combining Candrabindu Above
    'ౘ',  # Letter TTHA
    'ౙ',  # Letter DDA
    'ౚ',  # Letter RHA
    '౷',  # Vedic Tone
    '‘', '’', '“', '”', '%', '.', ';', '-', ',', '/', '\\', '_', '&',  # Common punctuation
    'G', 'P', 'S', 'e', 'l', 'n', 'r', 't', '\u200c', '\n' #Unwanted in the dataset
]

In [11]:
import re
chars_to_remove_regex = f'[{re.escape("".join(telugu_special_unwanted_characters))}]'

def remove_special_characters(batch):
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["transcription"])
    return batch

In [ ]:
train_dataset = train_dataset.map(remove_special_characters)
val_dataset = val_dataset.map(remove_special_characters)

In [ ]:
show_random_elements(train_dataset.remove_columns(["audio"]))

In [ ]:
def extract_all_chars(batch):
  all_text = " ".join(batch["transcription"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

vocab_train = train_dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=train_dataset.column_names)
vocab_test = val_dataset.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=val_dataset.column_names)

In [15]:
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]))

vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}

In [16]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

In [ ]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

In [18]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In [19]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", clean_up_tokenization_spaces=False)

In [20]:
repo_name = "wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs"

In [ ]:
tokenizer.push_to_hub(repo_name)

In [22]:
from transformers import Wav2Vec2FeatureExtractor

feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=False)

In [23]:
from transformers import Wav2Vec2Processor

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

In [ ]:
train_dataset[0]["audio"]

In [25]:
from datasets import Audio
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
val_dataset = val_dataset.cast_column("audio", Audio(sampling_rate=16_000))

In [ ]:
rand_int = random.randint(0, len(train_dataset))

print("Target text:", train_dataset[rand_int]["transcription"])
print("Input array shape:", train_dataset[rand_int]["audio"]["array"].shape)
print("Sampling rate:", train_dataset[rand_int]["audio"]["sampling_rate"])

In [27]:
def prepare_dataset(batch):
    audio = batch["audio"]

    # batched output is "un-batched"
    batch["input_values"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_values[0]
    batch["input_length"] = len(batch["input_values"])

    batch["labels"] = processor(text=batch["transcription"]).input_ids

    return batch

In [ ]:
train_dataset = train_dataset.map(prepare_dataset, remove_columns=train_dataset.column_names, num_proc = 4)
val_dataset = val_dataset.map(prepare_dataset, remove_columns=val_dataset.column_names, num_proc = 4)

In [29]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    """
    Data collator that will dynamically pad the inputs received.
    Args:
        processor (:class:`~transformers.Wav2Vec2Processor`)
            The processor used for proccessing the data.
        padding (:obj:`bool`, :obj:`str` or :class:`~transformers.tokenization_utils_base.PaddingStrategy`, `optional`, defaults to :obj:`True`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding index)
            among:
            * :obj:`True` or :obj:`'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
              sequence if provided).
            * :obj:`'max_length'`: Pad to a maximum length specified with the argument :obj:`max_length` or to the
              maximum acceptable input length for the model if that argument is not provided.
            * :obj:`False` or :obj:`'do_not_pad'` (default): No padding (i.e., can output a batch with sequences of
              different lengths).
    """

    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

In [30]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [32]:
import numpy as np

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

In [ ]:
from transformers import Wav2Vec2ForCTC

model = Wav2Vec2ForCTC.from_pretrained(
    'facebook/wav2vec2-large-xlsr-53',
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

In [34]:
model.freeze_feature_encoder()

In [35]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir=repo_name,
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  eval_strategy="steps",
  num_train_epochs=45,
  gradient_checkpointing=True,
  fp16=True,
  save_steps=600,
  eval_steps=400,
  logging_steps=400,
  learning_rate=3e-4,
  warmup_steps=400,
  save_total_limit=2,
  report_to="wandb",
  push_to_hub=True,
)

In [ ]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=processor.feature_extractor,
)

In [ ]:
trainer.train()

In [38]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/kaarthu2003/wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs/commit/396839db7d3e7cc55626f86bfc0765c6f4535128', commit_message='End of training', commit_description='', oid='396839db7d3e7cc55626f86bfc0765c6f4535128', pr_url=None, repo_url=RepoUrl('https://huggingface.co/kaarthu2003/wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs', endpoint='https://huggingface.co', repo_type='model', repo_id='kaarthu2003/wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("kaarthu2003/wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs")
model = AutoModelForCTC.from_pretrained("kaarthu2003/wav2vec2-final-learning-rate-3e-4_batch-size-16-45-epochs")

In [ ]:
import torch

# Move the model to the GPU
model.to("cuda")

def map_to_result(batch):
  with torch.no_grad():
    input_values = torch.tensor(batch["input_values"], device="cuda").unsqueeze(0)
    # Now the model and input_values are on the same device
    logits = model(input_values).logits

  pred_ids = torch.argmax(logits, dim=-1)
  batch["pred_str"] = processor.batch_decode(pred_ids)[0]
  batch["text"] = processor.decode(batch["labels"], group_tokens=False)

  return batch

results = val_dataset.map(map_to_result, remove_columns=val_dataset.column_names)

In [ ]:
print("Test WER: {:.3f}".format(wer_metric.compute(predictions=results["pred_str"], references=results["text"])))

In [42]:
import evaluate
cer_metric = evaluate.load("cer")

In [ ]:
print("Test CER: {:.3f}".format(cer_metric.compute(predictions=results["pred_str"], references=results["text"])))